# NB-20-ELIA · Jupyter companion

Calculations use the private Workbench API, pinned to version **1.43.0**. You need your individual API key and the private input bundle supplied for **NB-20-ELIA**.

1. Open this notebook in a fresh Python 3.12 or 3.13 kernel. Keep `notebook-methods-v1.43.0.zip` beside it when working locally; otherwise setup downloads the checksum-verified public helper archive.
2. Run the helper setup cell, then select your private input directory or ZIP. Local Jupyter can use `EWB_NOTEBOOK_INPUT_DIR` or `EWB_NOTEBOOK_INPUT_ZIP`. Colab uploads send inputs to your Google runtime; use local Jupyter when the inputs must stay on your computer.
3. Run the teaching cells in order. Inspect inputs, receipts, tables and plots locally. Results are saved beneath `data/processed/notebook-results`; save the notebook privately after execution.

The download contains no saved execution outputs. [Previously reviewed figures and results](https://sherandoresearch.github.io/power-markets-series/NB-20-ELIA.html) remain available in the static reader. A successful calculation validates its numerical contract, not a scientific claim. Saved results retain their original version; these notebooks request version 1.43.0 explicitly.

The static reader preserves previously reviewed results; rerunning this notebook creates new private execution evidence.

<!-- book-volume-context -->
## Where this fits in the volumes

This applies the temporal-graph discussion to observed Belgian border flows. Coverage is a partial border network, so the declared directed affinity cannot establish a complete European grid operator or causal relation.

- [Volume III, Chapter 5 — Temporal graph analysis](https://sherandoresearch.github.io/power-markets-series/vol3_spectral_grids.html#c5)
- [Volume V, Chapter 4 — Operators, coordinates, and representations](https://sherandoresearch.github.io/power-markets-series/vol5_representing_power_markets.html#c4)

In [ ]:
# Install the reviewed transport and notebook helpers. This cell makes no API calls.
import hashlib, importlib.metadata, io, json, os, pathlib, ssl, stat, subprocess, sys, tempfile, urllib.request, zipfile

if sys.version_info[:2] not in {(3, 12), (3, 13)}:
    raise RuntimeError("Use a Python 3.12 or 3.13 kernel for this reviewed release.")
helper_name = "notebook-methods-v1.43.0.zip"
expected_sha256 = "2bea8f592cae108b96369caded46f15631a60f1cd22d033cdd46905c3ce0ee18"
expected_files = json.loads('{"requirements.txt": "d3057422eeabe167fdc242f31c5038add6e0aa9aa73fb31227eb92b688cbb553", "src/ewb_methods_client/__init__.py": "faddf128f002243598cb56ab227945e50a2c6219453f24a2c633bf32c7ccf4ab", "src/methods_notebooks/__init__.py": "94e70a66bebea6221748a48192079f14297c552697aebc78205fd49128f9a345", "src/methods_notebooks/session.py": "86f788c33cc0db110ae1ae41f5cd4588af7029d8cd3aa935e491cabf6af40b3e", "src/methods_notebooks/setup.py": "17990abbd57a1472a44071834a7364604b452694605e6e4b19624ef8377e6d9f"}')
helper_path = pathlib.Path(os.environ.get("EWB_NOTEBOOK_HELPERS_ZIP", helper_name)).expanduser()
if helper_path.is_file():
    helper_bytes = helper_path.read_bytes()
else:
    helper_url = "https://github.com/sherandoresearch/power-markets-notebooks/releases/download/notebooks-v1.43.0/" + helper_name
    # Only the public, checksum-pinned transport helpers are downloaded here.
    # Inputs and API credentials are selected in the next cell.
    # Python.org macOS installs may lack a system CA bundle. Use the verified
    # certifi trust store when present; certificate and hostname checks stay on.
    try:
        import certifi
    except ImportError:
        tls_context = ssl.create_default_context()
    else:
        tls_context = ssl.create_default_context(cafile=certifi.where())
    with urllib.request.urlopen(helper_url, timeout=90, context=tls_context) as response:
        helper_bytes = response.read(1_000_001)
if len(helper_bytes) > 1_000_000 or hashlib.sha256(helper_bytes).hexdigest() != expected_sha256:
    raise ValueError("Notebook helper release checksum mismatch")
with zipfile.ZipFile(io.BytesIO(helper_bytes)) as helper_zip:
    members = helper_zip.infolist()
    if len(members) != len(expected_files) or {m.filename for m in members} != set(expected_files):
        raise ValueError("Unexpected helper archive contents")
    if any(stat.S_IFMT(m.external_attr >> 16) not in (0, stat.S_IFREG) or
           m.flag_bits & 1 or m.file_size > 100_000 for m in members):
        raise ValueError("Unsupported helper archive member")
    helper_contents = {name: helper_zip.read(name) for name in expected_files}
if any(hashlib.sha256(value).hexdigest() != expected_files[name] for name, value in helper_contents.items()):
    raise ValueError("Notebook helper member checksum mismatch")
requirements = helper_contents["requirements.txt"].decode().splitlines()
missing = []
for requirement in requirements:
    package, version = requirement.split("==")
    try:
        current_version = importlib.metadata.version(package)
    except importlib.metadata.PackageNotFoundError:
        current_version = None
    if current_version != version:
        if package in sys.modules:
            raise RuntimeError("Restart in a fresh kernel before installing " + requirement)
        missing.append(requirement)
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *missing])
for requirement in requirements:
    package, version = requirement.split("==")
    if importlib.metadata.version(package) != version:
        raise RuntimeError("Dependency version mismatch: " + package)
if any(name == "methods_notebooks" or name.startswith("methods_notebooks.") or
       name == "ewb_methods_client" or name.startswith("ewb_methods_client.") for name in sys.modules):
    raise RuntimeError("Restart the kernel before rerunning helper setup.")
helper_parent = pathlib.Path.cwd() / "data/processed/notebook-helpers"
helper_parent.mkdir(parents=True, exist_ok=True, mode=0o700)
helper_root = pathlib.Path(tempfile.mkdtemp(prefix="v1.43.0-", dir=helper_parent))
for name, value in helper_contents.items():
    target = helper_root / name
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_bytes(value)
sys.path.insert(0, str(helper_root / "src"))
del helper_bytes, helper_contents
print("Notebook helpers verified. Calculations use Workbench 1.43; inputs and results stay private.")


In [ ]:
from methods_notebooks.setup import configure_notebook
setup_status = configure_notebook('NB-20-ELIA', require_key=True)
print(setup_status)


# NB-20-ELIA — Observed border flows and direction
An observed-data companion to NB-20, linked to Volume III §5 and Volume IV §11.
**Question:** what does directional structure mean when the observations cover
only Belgium's borders? We use real Elia quarter-hour flows from 10–12 October
2026, served through EWB, with no generated or substituted observations.

Audit signs and coverage, integrate MW over time, build a declared gross-flow
relation, then test what changes under coordinate reversal, transposition, and
different daily windows. This notebook reuses the existing directed-generator
implementation. It does not infer an internal grid or causal propagation.

In [ ]:
NOTEBOOK_ID = 'NB-20-ELIA'

import copy, json
from datetime import datetime
from types import SimpleNamespace
import numpy as np
from methods_notebooks.session import Session, MethodsError, setup_plot
session = Session(NOTEBOOK_ID, version='1.43.0')
inputs = session.inputs
responses, execution_plan = {}, []
def execute(name, method, request, parents=()):
    request = copy.deepcopy(request)
    if parents:
        request['parent_runs'] = [{k:responses[p]['receipt'][k] for k in
            ('run_id','scientific_method_id','method_version','result_sha256')} for p in parents]
    response = session.execute(method, request)
    responses[name] = response
    execution_plan.append(dict(name=name,method=method,request=request,parents=list(parents),rest_receipt=response['receipt']))
    (session.output/'execution-plan.json').write_text(json.dumps(execution_plan,indent=2,allow_nan=False)+'\n')
    return response['result']
def reject(method, request):
    try:
        session.execute(method, request)
    except MethodsError as exc:
        assert str(exc) == 'Methods request refused (HTTP 422)'
        print('Expected remote rejection:',method,str(exc))
    else:
        raise AssertionError('Invalid retained-input mutation accepted')
plt=setup_plot()

nodes,borders=inputs['nodes'],inputs['borders']
print('Local raw inspection:',inputs['raw_rows'][:1])
x=dict(audit=inputs['audit'],times=[datetime.fromisoformat(t) for t in inputs['times']],
       flow=np.asarray(inputs['flow']),windows=[])
def generators(name, mean, policy, parent, control='observed'):
    original=np.asarray(mean['values'],dtype=float)
    mask=np.isfinite(original)
    # Null off-support entries become excluded feature coordinates only.
    expected=np.zeros((6,6),dtype=bool);expected[0,1:]=True;expected[1:,0]=True
    assert np.array_equal(mask,expected)
    a=np.where(mask,original,0.)
    if control=='transpose':a=a.T
    elif control=='symmetrized':a=(a+a.T)/2
    assert control in ('observed','transpose','symmetrized')
    request=dict(inputs=dict(directed_affinity=dict(node_ids=nodes,values=a.tolist(),support_mask=mask.tolist(),
        row_convention='origin_row_destination_column',affinity_unit='MW',
        observation_window=dict(kind='utc_interval',start=policy['start'],end=policy['end']),
        support_policy='unobserved_excluded',self_loop_policy='reject',
        source=dict(reference_id='NB-20-ELIA:'+parent,artifact_sha256=responses[parent]['receipt']['result_sha256'],
            construction='Complete-day gross-flow relation; excluded unobserved pairs; '+control+' algebraic control')),
        generator_parameters=dict(theta=1,bandwidth=1,bandwidth_unit='dimensionless',
            normalization='symmetric_source_density_power',zero_degree_policy='reject',kernel_convention='source_zero_row_sum')),
        parameters={},context=dict(input_origin='Retained Elia observations; '+control+'; no physical or causal identification'))
    result=execute(name,'directed_generators',request,[parent])['directed_generator_bundle']
    return a,np.asarray(result['geometry']),np.asarray(result['drift'])
for day in range(3):
    name=f'day{day}_complete_window_mean'
    req=inputs['requests'][name]
    result=execute(name,'border_flow_affinity',req)
    mean=result['directed_affinity'];policy=req['inputs']['affinity_construction_policy']
    assert all(b['coverage_fraction']==1 for b in result['support_ledger']['borders'])
    energy=execute(f'day{day}_energy','border_flow_affinity',inputs['requests'][f'day{day}_energy'])['directed_affinity']
    a,g,r=generators(f'day{day}_generators',mean,policy,name)
    ev=np.asarray(energy['values'],dtype=float)
    assert energy['unit']=='MWh' and mean['unit']=='MW'
    np.testing.assert_allclose(ev[np.isfinite(ev)],24*a[np.isfinite(ev)],atol=1e-8)
    x['windows'].append(dict(day=policy['start'][:10],affinity=a,geometry=g,drift=r,
        export_mwh=ev[0,1:],import_mwh=ev[1:,0],net_mwh=ev[0,1:]-ev[1:,0],gross_mean_mw=a[0,1:]+a[1:,0]))
x['changes']=[dict(days=x['windows'][i]['day']+' to '+x['windows'][i+1]['day'],
    geometry_relative=float(np.linalg.norm(x['windows'][i+1]['geometry']-x['windows'][i]['geometry'])/np.linalg.norm(x['windows'][i]['geometry'])),
    drift_relative=float(np.linalg.norm(x['windows'][i+1]['drift']-x['windows'][i]['drift'])/np.linalg.norm(x['windows'][i]['drift']))) for i in range(2)]
print(json.dumps(x['audit'],indent=2))

## 1. Define the observed panel and its clock
The retained snapshot is `elia-20261015T024947Z-23370735`. All 1,440 rows
are present: 288 quarter-hours × five borders, with unique time/border keys,
finite MW values, PT15M resolution, and five stable series identifiers. The
selected window ends at 23:45 UTC on 12 October; its final interval ends at
midnight. It is a dated descriptive sample, not a current operating picture.

The retained dataset contract defines positive MW as **exports from Belgium**.
Border labels DE, FR, GB, LU and NL are counterparty control areas. They are not
five individual AC lines: some are aggregate borders or interconnectors. Keep
these exact source coordinates instead of silently merging DE and LU.

Event time and collector receipt are separate. Elia supplies no source-publication
timestamp in this feed, and value-based vintages preserve first receipt of a
value. Latest data received later cannot establish what was available at delivery.
The source is Elia's `ods026` feed, through the retained EWB adapter contract.

In [ ]:
colors=['#88C0D0','#EBCB8B','#A3BE8C','#B48EAD','#D08770']
fig,ax=plt.subplots(figsize=(11,4))
for j,border in enumerate(borders):ax.plot(x['times'],x['flow'][:,j],label=border,color=colors[j],linewidth=1)
ax.axhline(0,color='#ECEFF4',linewidth=.7)
ax.set(title='Observed Belgian border flows',ylabel='MW · positive = export from BE',xlabel='UTC');ax.legend(ncol=5)
fig.autofmt_xdate();fig.tight_layout();plt.show()

## 2. Gross direction and net energy are different summaries
For each complete UTC day, export MWh = Σ max(f,0) × 0.25 h, import MWh =
Σ max(−f,0) × 0.25 h, and signed net MWh = exports − imports. These are
directional integrals of the reported quarter-hour series, not simultaneous
counterflows on a particular line. A border can reverse during the day.

Do not average the signed flow first and then take its positive part: that loses
reversals. Three complete UTC days avoid DST-duration ambiguity in this example.
No price, commercial schedule, loss adjustment, or domestic generation/load is
included. Summing these five borders is only a boundary-flow summary.

In [ ]:
print('UTC day | border | export MWh | import MWh | signed net MWh')
for w in x['windows']:
    np.testing.assert_allclose(w['export_mwh']-w['import_mwh'],w['net_mwh'],atol=1e-9)
    for j,border in enumerate(borders):print(w['day'],border,*[round(w[k][j],2) for k in ('export_mwh','import_mwh','net_mwh')],sep=' | ')
fig,axes=plt.subplots(1,3,figsize=(13,4),sharey=True)
for ax,w in zip(axes,x['windows']):
    ax.bar(np.arange(5)-.18,w['export_mwh']/1000,width=.36,label='Export',color='#88C0D0')
    ax.bar(np.arange(5)+.18,w['import_mwh']/1000,width=.36,label='Import',color='#B48EAD')
    ax.set_xticks(range(5),borders);ax.set(title=w['day'],xlabel='Border')
axes[0].set_ylabel('Integrated reported flow · GWh');axes[0].legend();fig.tight_layout();plt.show()

## 3. A declared feature graph, not a recovered physical network
The six coordinates are BE and the five counterparties. On each selected BE–j
edge, set a_BE,j = mean(max(f_j,0)) and a_j,BE = mean(max(−f_j,0)), in MW.
This square, nonnegative relation can be passed to NB-20's directed operators.
**Unobserved counterparty-to-counterparty edges are excluded from this feature
graph. Their matrix zeros do not assert zero physical flow.**

With θ=1 and bandwidth=1, q is the average in/out strength, B_ij=a_ij/(q_i q_j),
S=(B+Bᵀ)/2, A=(B−Bᵀ)/2, d=S1. The generators are
G=D⁻¹(S−diag(d)) and R=D⁻¹(A−diag(A1)). Bandwidth is a normalization choice,
not a physical time unit. All selected nodes have positive degree; a zero-degree
window would fail rather than receive an invented positive edge.

In [ ]:
a=x['windows'][0]['affinity'];g=x['windows'][0]['geometry'];r=x['windows'][0]['drift']
from matplotlib.colors import LinearSegmentedColormap
cmap=LinearSegmentedColormap.from_list('nord_diverging',['#5E81AC','#ECEFF4','#BF616A'])
fig,axes=plt.subplots(1,3,figsize=(13,4))
for ax,array,title in zip(axes,[a,g,r],['Gross-flow feature (MW)','Geometry generator','Drift generator']):
    bound=np.abs(array).max();im=ax.imshow(array,cmap=cmap,vmin=-bound,vmax=bound)
    ax.set_xticks(range(6),nodes);ax.set_yticks(range(6),nodes);ax.set(title=title,xlabel='Destination coordinate',ylabel='Source coordinate')
    fig.colorbar(im,ax=ax,fraction=.046)
fig.tight_layout();plt.show()

## 4. Coordinate reversal, directional reversal, and window sensitivity
Reversing every edge coordinate **and** its signed flow leaves the observed
directed relation unchanged. Transposing the affinity instead reverses that
relation: G is unchanged and R changes sign. Transposition is an algebraic
control on retained observations; it is not an additional observed market day.
Symmetrization discards the directional component, giving zero drift.

The geometry also stays constant across these three daily windows. This is
**structural**, not evidence that the European network was stable. On a star
with θ=1, q_j equals half the gross strength of leaf j, while q_BE is half the
total strength. Every normalized symmetric BE–j edge therefore has the same
value. G sends each leaf to BE with unit weight and BE equally to its five leaves,
independent of daily gross magnitudes. Drift retains directional imbalance.

In [ ]:
base='day0_complete_window_mean'
req=copy.deepcopy(inputs['requests'][base])
for row in req['inputs']['border_flow_records']:
    row['source'],row['sink']=row['sink'],row['source'];row['value_mw']=-row['value_mw']
reversed_result=execute('reversed_coordinates','border_flow_affinity',req)['directed_affinity']
np.testing.assert_allclose(np.asarray(reversed_result['values'],dtype=float),np.asarray(responses[base]['result']['directed_affinity']['values'],dtype=float),equal_nan=True,atol=1e-12)
mean=responses[base]['result']['directed_affinity'];policy=req['inputs']['affinity_construction_policy']
_,gt,rt=generators('transpose',mean,policy,base,'transpose')
_,gs,rs=generators('symmetrized',mean,policy,base,'symmetrized')

np.testing.assert_allclose(g,gt,atol=1e-12);np.testing.assert_allclose(r,-rt,atol=1e-12)
np.testing.assert_allclose(g,gs,atol=1e-12);np.testing.assert_allclose(rs,0,atol=1e-12)
np.testing.assert_allclose(g.sum(1),0,atol=1e-12);np.testing.assert_allclose(r.sum(1),0,atol=1e-12)
np.testing.assert_allclose(g[0,1:],np.full(5,1/5),atol=1e-12)
np.testing.assert_allclose(g[1:,0],1,atol=1e-12)
print(json.dumps(x['changes'],indent=2))
fig,axes=plt.subplots(1,2,figsize=(12,4))
for i,w in enumerate(x['windows']):
    axes[0].plot(range(5),w['gross_mean_mw'],marker='o',label=w['day'],color=colors[i])
    axes[1].plot(range(6),np.diag(w['drift']),marker='o',label=w['day'],color=colors[i])
axes[0].set_xticks(range(5),borders);axes[0].set(title='Gross strength changes',ylabel='Mean absolute MW')
axes[1].set_xticks(range(6),nodes);axes[1].set(title='Directional drift changes',ylabel='Recentered drift diagonal')
for ax in axes:ax.legend(fontsize=8)
fig.tight_layout();plt.show()

## 5. What this boundary cannot identify
The selected feature graph has six nodes, five edges, and zero cycle dimension.
A Hodge residual on this tree cannot reveal omitted European loop flows: any
edge vector is gradient-representable on a tree. We therefore do not present a
zero cycle residual as a validation of physical power-flow equations.

Exercise: can constant G and changing R prove a stable topology with changing
causal influence? No. The constant geometry follows from the feature construction;
directional drift is a descriptive statistic of observed flows. Internal Belgian
topology, neighboring-country edges, susceptances, outages, commercial transfers,
a manifold embedding, and causal direction remain unidentified.
To extend this sample, acquire and validate those observations rather than filling
unobserved edges or generating hypothetical flows. Historical GB coverage alone
also cannot establish five-border completeness before this dated window.

In [ ]:
print('Selected graph cycle dimension:',x['audit']['feature_graph_cycle_dimension'])
assert x['audit']['feature_graph_cycle_dimension']==0

In [ ]:
bad=copy.deepcopy(inputs['requests']['day0_complete_window_mean']);bad['inputs']['border_flow_records'].pop()
reject('border_flow_affinity',bad)
bad=copy.deepcopy(inputs['requests']['day0_complete_window_mean']);row=copy.deepcopy(bad['inputs']['border_flow_records'][0]);row['source'],row['sink']=row['sink'],row['source'];row['value_mw']=-row['value_mw'];bad['inputs']['border_flow_records'].append(row)
reject('border_flow_affinity',bad)
(session.output/'consumer-summary.json').write_text(json.dumps(dict(audit=x['audit'],changes=x['changes'],
    windows=[{k:v.tolist() if isinstance(v,np.ndarray) else v for k,v in w.items()} for w in x['windows']]),indent=2,allow_nan=False)+'\n')
summary=session.finish({'observed_only':True,'complete_five_border_panel':True,'signed_energy_identity':True,
    'coordinate_reversal_invariant':True,'transpose_controls':True,'star_geometry_limit_disclosed':True,'new_synthetic_data':False},
    {'window':'2026-10-10 to 2026-10-12 UTC','theta':1,'bandwidth':1,'graph':'selected Belgian border star only'})